# Simplified Transform Analysis
## Color transformation

Now let's see how spectral chrarcteristics of the measurement channel
affects transform values.

In [ ]:
from vsopy.util import ValErr
from measurement import SimpleMeasurement
from experiment import Experiment
from sequence import StarSequence
import numpy as np
from astropy.table import QTable

SEED = 42

class SkewedMeasurement(SimpleMeasurement):

    def __init__(self, rng, params: SimpleMeasurement.Params, skew: tuple[float, float]):
        super().__init__(rng, params)
        self.skew = skew

    def generate_magnitudes(self, seq: StarSequence) -> QTable:
        measured = super().generate_magnitudes(seq)
        ci = measured[seq.stars['band'] == seq.bands[0]] - measured[seq.stars['band'] == seq.bands[1]]
        measured[seq.stars['band'] == seq.bands[0]] += ci * self.skew[0]
        measured[seq.stars['band'] == seq.bands[1]] += ci * self.skew[1]
        return measured


EXPERIMENT = Experiment(SEED)

SEQUENCE = EXPERIMENT.generate_sequence(
    Experiment.GenParams(
        num_stars=20,
        mag=(10.0, 15.0),
        ci=(-0.5, 1.0),
        uncert=ValErr(0.05, 0.05)))
SEQUENCE.stars

In [ ]:
from util import simple_transform_stats

SKEW = np.linspace(-0.1, 0.1, 11)
TR_STATS = QTable(
    [ simple_transform_stats(
        EXPERIMENT.create_transforms(
            SEQUENCE,
            EXPERIMENT.create_measurement(
                SkewedMeasurement,
                params = SimpleMeasurement.Params(
                    offset=1.0,
                    scatter=0.05,
                    uncert=ValErr(0.03, 0.03)),
                skew=(s_A, s_B)),
                num_measurements=200))
               | {'skew A': s_A, 'skew B': s_B} for s_A in SKEW for s_B in SKEW]
    )
TR_STATS.to_pandas()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

X = np.array(TR_STATS['skew A']).reshape(len(SKEW), len(SKEW))
Y = np.array(TR_STATS['skew B']).reshape(len(SKEW), len(SKEW))

PREFIX = ['Tab', 'Ta', 'Tb']
SUFFIX = ['', '.std', '.err']
FIG, AXS = plt.subplots(len(PREFIX), len(SUFFIX), figsize=(10.24, 10.24), constrained_layout=True)


def plot_contour(ax, column):
    z = np.array(TR_STATS[column]).reshape(len(SKEW), len(SKEW))
    #ax = FIG.add_subplot(GS[row, col])
    ax.set_box_aspect(1)
    cont = ax.contourf(X, Y, z, levels=20, cmap='viridis')
    cb = FIG.colorbar(cont, ax=ax, fraction=0.046, pad=0.04, orientation="horizontal")
    cb.locator = mpl.ticker.MaxNLocator(nbins=5)
    cb.update_ticks()
    if row == len(PREFIX) - 1:
        ax.set_xlabel('Skew A')
    if col == 0:
        ax.set_ylabel('Skew B')
    ax.set_title(column)
    return cont

for col in range(len(SUFFIX)):
    for row in range(len(PREFIX)):
        plot_contour(AXS[row, col], f'{PREFIX[row]}{SUFFIX[col]}')

plt.show()